In [3]:
import sqlite3

conn = sqlite3.connect("../database/inventory.db")
cursor = conn.cursor()

In [4]:
def view_inventory():
    cursor.execute("SELECT * FROM Inventory;")
    rows = cursor.fetchall()
    
    for row in rows:
        print(f"Product: {row[1]} | Qty: {row[2]}")

view_inventory()

Product: Reagent X | Qty: 50
Product: Test Kit A | Qty: 10
Product: Buffer Solution B | Qty: 0
Product: Chemical Compound C | Qty: 75
Product: Diagnostic Kit D | Qty: 5
Product: Enzyme Mix E | Qty: 100
Product: Culture Media F | Qty: 20
Product: Protein Sample G | Qty: 60
Product: Antibody H | Qty: 8
Product: Solvent I | Qty: 200
Product: Vaccine Component J | Qty: 0
Product: Lab Kit K | Qty: 30
Product: pH Buffer L | Qty: 12
Product: Growth Factor M | Qty: 90
Product: Sterile Filter N | Qty: 3


In [21]:
def get_non_empty(prompt):
    while True:
        value = input(prompt).strip()
        if value:
            return value
        else: 
            print("Input cannot be empty.")

In [23]:
def get_int(prompt):
    while True:
        value = input(prompt)
        try:
            return int(value)
        except ValueError:
            print("Please enter a valid number.")

In [25]:
def low_stock_items():
    cursor.execute("""
        SELECT item_name, quantity, reorder_point
        FROM Inventory
        WHERE quantity <= reorder_point
        """)
    rows = cursor.fetchall()
    print("Low Stock Items: ")
    for row in rows:
        print(f"Product: {row[0]} | Qty: {row[1]} | RO Point: {row[2]}")

In [6]:
def inventory_with_suppliers():
    cursor.execute("""
        SELECT i.item_name, i.quantity, s.supplier_name
        FROM Inventory i
        JOIN Supplier s ON i.supplier_id = s.supplier_id;
    """)
    
    rows = cursor.fetchall()
    
    for row in rows:
        print(f"Product: {row[0]} | Qty: {row[1]} | Supplier: {row[2]}")

inventory_with_suppliers()

Product: Reagent X | Qty: 50 | Supplier: PharmaSupply Co
Product: Test Kit A | Qty: 10 | Supplier: BioLab Inc
Product: Buffer Solution B | Qty: 0 | Supplier: MedChem Solutions
Product: Chemical Compound C | Qty: 75 | Supplier: PharmaSupply Co
Product: Diagnostic Kit D | Qty: 5 | Supplier: BioLab Inc
Product: Enzyme Mix E | Qty: 100 | Supplier: LabSource LLC
Product: Culture Media F | Qty: 20 | Supplier: MedChem Solutions
Product: Protein Sample G | Qty: 60 | Supplier: PharmaSupply Co
Product: Antibody H | Qty: 8 | Supplier: BioLab Inc
Product: Solvent I | Qty: 200 | Supplier: LabSource LLC
Product: Vaccine Component J | Qty: 0 | Supplier: MedChem Solutions
Product: Lab Kit K | Qty: 30 | Supplier: BioLab Inc
Product: pH Buffer L | Qty: 12 | Supplier: PharmaSupply Co
Product: Growth Factor M | Qty: 90 | Supplier: LabSource LLC
Product: Sterile Filter N | Qty: 3 | Supplier: MedChem Solutions


In [ ]:
def search_inventory():
    keyword = input("Search inventory by product: ")
    cursor.execute("""
        SELECT item_name, quantity, reorder_point
        FROM Inventory
        WHERE item_name LIKE ?
        """, ("%" + keyword + "%",))

    rows = cursor.fetchall()
    if len(rows) == 0:
        print("No matching results.")
    else:
        for row in rows:
            print(f"Product: {row[0]} | Qty: {row[1]} | RO Point: {row[2]}")
    
search_inventory()

In [26]:
def select_supplier():
    while True:
        cursor.execute("SELECT supplier_id, supplier_name FROM Supplier")
        suppliers = cursor.fetchall()
        supplier_ids = [s[0] for s in suppliers]
        
        print("Suppliers:")
        for s in suppliers:
            print(f"{s[0]}. {s[1]}")

        print("\nOptions:")
        print("A - Add new supplier")
        print("Q - Cancel")

        choice = input("Enter supplier ID or choose an option: ").strip()

        if choice.lower() == 'q':
            print("Exiting program.")
            return None

        if choice.lower() == 'a':
            add_supplier()
            continue
    
        try:
            choice = int(choice)
        except ValueError:
            print("Invalid input.")
            continue
    
        if choice in supplier_ids:
            return choice
        else:
            print("Invalid supplier ID.")
select_supplier()

Suppliers:
1. PharmaSupply Co
2. BioLab Inc
3. MedChem Solutions
4. LabSource LLC

Options:
A - Add new supplier
Q - Cancel


Enter supplier ID or choose an option:  q


Exiting program.


In [ ]:
def select_location():
    cursor.execute("SELECT location_id, location_name FROM Location")
    locations = cursor.fetchall()
    location_ids = [l[0] for l in locations]

    while True:
        print("Locations:")
        for l in locations:
            print(f"{l[0]}. {l[1]}")

        choice = input("Enter location ID or type 'q' to exit: ")

        if choice.lower() == 'q':
            print("Exiting program.")
            return None
        
        try:
            choice = int(choice)
        except ValueError:
            print("Invalid input. Please enter a number.")
            continue
    
    
        if choice in location_ids:
            return choice
        else:
            print("Invalid location ID.")
        

select_location()

In [27]:
def add_product():
    new_product = get_non_empty("Add product: ")
    new_qty = get_int("Add Quantity: ")
    new_ro = get_int("Add Reorder Point: ")
    supplier_id = select_supplier()
    if supplier_id is None:
        return
    location_id = select_location()
    if location_id is None:
        return

    if new_qty <= new_ro:
        status = "Low Stock"
    else:
        status = "In Stock"
    
    cursor.execute("""
    INSERT INTO Inventory 
    (item_name, quantity, reorder_point, supplier_id, location_id, status)
    VALUES ( ?, ?, ?, ?, ?, ?)
    """, (new_product, new_qty, new_ro, supplier_id, location_id, status))

    conn.commit()

    print("Product Added Successfully!")
add_product()

Add product:  Test
Add Quantity:  3
Add Reorder Point:  2


Suppliers:
1. PharmaSupply Co
2. BioLab Inc
3. MedChem Solutions
4. LabSource LLC

Options:
A - Add new supplier
Q - Cancel


Enter supplier ID or choose an option:  q


Exiting program.


In [ ]:
def add_supplier():
    new_supplier = get_non_empty("Enter new supplier: ")

    cursor.execute("""
    INSERT INTO Supplier
    (supplier_name)
    VALUES (?)
    """, (new_supplier,))

    conn.commit()

    print("Supplier added successfully!")

In [7]:
cursor.execute("""
SELECT * FROM Inventory
""")
cursor.fetchall()

[(1, 'Reagent X', 50, 20, 'Available', 1, 1),
 (2, 'Test Kit A', 10, 15, 'Low Stock', 2, 2),
 (3, 'Buffer Solution B', 0, 10, 'Out of Stock', 3, 3),
 (4, 'Chemical Compound C', 75, 30, 'Available', 1, 1),
 (5, 'Diagnostic Kit D', 5, 10, 'Low Stock', 2, 4),
 (6, 'Enzyme Mix E', 100, 40, 'Available', 4, 2),
 (7, 'Culture Media F', 20, 25, 'Low Stock', 3, 3),
 (8, 'Protein Sample G', 60, 20, 'Available', 1, 4),
 (9, 'Antibody H', 8, 12, 'Low Stock', 2, 1),
 (10, 'Solvent I', 200, 50, 'Available', 4, 2),
 (11, 'Vaccine Component J', 0, 25, 'Out of Stock', 3, 3),
 (12, 'Lab Kit K', 30, 10, 'Available', 2, 4),
 (13, 'pH Buffer L', 12, 15, 'Low Stock', 1, 1),
 (14, 'Growth Factor M', 90, 30, 'Available', 4, 2),
 (15, 'Sterile Filter N', 3, 10, 'Low Stock', 3, 3)]

In [ ]:
def delete_product():
    deleted_product = get_non_empty("Enter the product name to be deleted: ")

    cursor.execute("""
        SELECT * FROM Inventory WHERE item_name = ?
    """, (deleted_product,))

    if cursor.fetchone() is None:
        print("Product not found.")
        return

    confirm = input(f"Delete '{deleted_product}'? (y/n): ")

    if confirm.lower() != 'y':
        print("Delete cancelled.")
        return

    cursor.execute("""
    DELETE FROM Inventory
    WHERE item_name = ?
    """, (deleted_product,))

    conn.commit()

    print("Product Successfully Deleted!")
delete_product()

In [52]:
def update_qty():
    product_name = get_non_empty("Enter the product you want to update: ")
    new_qty = get_int("Enter updated quantity: ")

    cursor.execute("""
        SELECT reorder_point 
        FROM Inventory 
        WHERE item_name = ?
    """, (product_name,))

    result = cursor.fetchone()

    if result is None:
        print("Product not found.")
        return

    reorder_point = result[0]
    
    if new_qty <= reorder_point:
        status = "low stock"
    else:
        status = "in stock"
    
    cursor.execute("""
    UPDATE Inventory
    SET quantity = ?, status = ?
    WHERE item_name = ?
    """, (new_qty, status, product_name))

    conn.commit()

    print("Quantity Successfully Updated!")

_IncompleteInputError: incomplete input (3965011533.py, line 5)

In [ ]:
menu_choices = {
    "1": view_inventory,
    "2": low_stock_items,
    "3": inventory_with_suppliers,
    "4": search_by_product,
    "5": add_product,
    "6": delete_product,
    "7": update_qty,
}

def menu():
    print("\nInventory System")
    print("1. View Inventory")
    print("2. Low Stock Items")
    print("3. Supplier View")
    print("4. Search By Product")
    print("5. Add Product")
    print("6. Delete Product")
    print("7. Update Quantity")
    print("8. Exit")
    
    choice = input("Enter your choice: ").strip()
    return choice

while True:
    choice = menu()

    if choice == "8":
        print("Exiting program...")
        break
    elif choice in menu_choices:
        menu_choices[choice]()
    else:
        print("Invalid choice. Please select 1-8.")

    if choice == "1":
        view_inventory()
    elif choice == "2":
        low_stock_items()
    elif choice == "3":
        inventory_with_suppliers()
    elif choice == "4":
        search_inventory()
    elif choice == "5":
        add_product()
    elif choice == "6":
        delete_product()
    elif choice == "7":
        update_qty()
    elif choice == "8":
        print("Exiting program...")
        break
    else:
        print("Invalid choice. Please select 1-8.")